# 05 — Governed Boundary Contracts

This notebook demonstrates how to declare and enforce **governed portal
contracts** — the ontology, ownership, classification, and alignment
metadata that gives a boundary its authority, not just its structural shape.

SHACL shapes validate *what* can cross a membrane. This notebook shows
how to declare *under whose authority*, *governed by which ontology*,
*conforming to which standard*, and *at what classification level* the
data crosses — and how the library surfaces violations when those
governance declarations conflict.

Concepts exercised:

- `cga:AlignmentHolon` — a dedicated holon containing the vocabulary
  mapping between two ontologies and its provenance
- `cga:DataDomain` with `cga:domainSteward` and `cga:domainPolicy` —
  governance scope and accountability
- `cga:dataSteward` / `cga:dataOwner` — people responsible for a
  holon's boundary contract
- `cga:dataClassification` — sensitivity level, enforced at portal
  boundaries by `cga:PortalClassificationShape`
- `cga:sourceOfTruthFor` / `cga:repositoryOfTruthFor` — authority
  declarations
- `cga:usesAlignment` on a portal — links the traversal to its
  semantic justification

## Setup — Two departments, different ontologies

A defense R&D organization has two departments. The Intelligence
department uses a CCO-derived ontology for sensor tracks. The Operations
department uses Schema.org for planning artifacts. A portal between
them translates CCO tracks into Schema.org events.

The translation is not just a CONSTRUCT query — it carries governance
metadata declaring who owns the contract, which ontologies govern each
side, what classification levels are in play, and which alignment
holon justifies the mapping.

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# ── Organization root ──

ds.add_holon("urn:holon:org", "Defense R&D Organization")
ds.add_interior("urn:holon:org", '''
    @prefix schema: <https://schema.org/> .
    <urn:org:defense-rd> a schema:Organization ;
        schema:name "Defense R&D Organization" .
''')

# ── Data domains (governance scope) ──

ds.add_holon("urn:holon:domain-isr", "ISR Domain",
             member_of="urn:holon:org")
ds.add_interior("urn:holon:domain-isr", '''
    @prefix cga:    <urn:holonic:ontology:> .
    @prefix schema: <https://schema.org/> .

    <urn:domain:isr> a cga:DataDomain ;
        rdfs:label "Intelligence, Surveillance, and Reconnaissance" ;
        cga:domainSteward <urn:person:j-chen> ;
        cga:domainPolicy  <urn:policy:isr-data-handling> .

    <urn:person:j-chen> a schema:Person ;
        schema:name "J. Chen" ;
        schema:jobTitle "ISR Data Steward" .
''')

ds.add_holon("urn:holon:domain-ops", "Operations Domain",
             member_of="urn:holon:org")
ds.add_interior("urn:holon:domain-ops", '''
    @prefix cga:    <urn:holonic:ontology:> .
    @prefix schema: <https://schema.org/> .

    <urn:domain:ops> a cga:DataDomain ;
        rdfs:label "Operations Planning" ;
        cga:domainSteward <urn:person:m-park> ;
        cga:domainPolicy  <urn:policy:ops-data-handling> .

    <urn:person:m-park> a schema:Person ;
        schema:name "M. Park" ;
        schema:jobTitle "Operations Data Steward" .
''')

print(f"Holons: {len(ds.list_holons())}")

## Department holons with governance metadata

Each department declares:
- Which ontology governs its interior data
- Who stewards it
- What classification level its data carries
- Which data domain it is the source of truth for

In [ ]:
# ── Intelligence department (CCO-governed) ──

ds.add_holon("urn:holon:intel", "Intelligence Dept",
             member_of="urn:holon:domain-isr")
ds.add_interior("urn:holon:intel", '''
    @prefix cga:    <urn:holonic:ontology:> .
    @prefix schema: <https://schema.org/> .
    @prefix dcterms: <http://purl.org/dc/terms/> .

    <urn:holon:intel> cga:dataSteward <urn:person:j-chen> ;
        cga:dataOwner          <urn:person:a-kim> ;
        cga:dataClassification cga:Secret ;
        cga:sourceOfTruthFor   <urn:domain:isr> ;
        dcterms:conformsTo     <urn:ontology:cco-sensor> .

    <urn:person:a-kim> a schema:Person ;
        schema:name "A. Kim" ;
        schema:jobTitle "Intel Division Chief" .
''')

# Sensor track data (CCO-style)
ds.add_interior("urn:holon:intel", '''
    @prefix cco: <urn:cco:> .

    <urn:track:001> a cco:SensorTrack ;
        cco:latitude     34.05 ;
        cco:longitude   -118.25 ;
        cco:confidence   0.92 ;
        cco:trackSource  "SIGINT" ;
        cco:timestamp    "2026-04-18T14:30:00Z"^^<http://www.w3.org/2001/XMLSchema#dateTime> .

    <urn:track:002> a cco:SensorTrack ;
        cco:latitude     35.68 ;
        cco:longitude    139.69 ;
        cco:confidence   0.78 ;
        cco:trackSource  "GEOINT" ;
        cco:timestamp    "2026-04-18T14:35:00Z"^^<http://www.w3.org/2001/XMLSchema#dateTime> .
''', graph_iri="urn:holon:intel/interior/tracks")

ds.add_boundary("urn:holon:intel", '''
    @prefix sh:  <http://www.w3.org/ns/shacl#> .
    @prefix cco: <urn:cco:> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    <urn:shapes:SensorTrackShape> a sh:NodeShape ;
        sh:targetClass cco:SensorTrack ;
        sh:property [ sh:path cco:latitude ;    sh:minCount 1 ; sh:datatype xsd:decimal ] ;
        sh:property [ sh:path cco:longitude ;   sh:minCount 1 ; sh:datatype xsd:decimal ] ;
        sh:property [ sh:path cco:confidence ;  sh:minCount 1 ; sh:datatype xsd:decimal ] ;
        sh:property [ sh:path cco:trackSource ; sh:minCount 1 ; sh:datatype xsd:string ] ;
        sh:property [ sh:path cco:timestamp ;   sh:minCount 1 ; sh:datatype xsd:dateTime ] .
''')

# ── Operations department (Schema.org-governed) ──

ds.add_holon("urn:holon:ops", "Operations Dept",
             member_of="urn:holon:domain-ops")
ds.add_interior("urn:holon:ops", '''
    @prefix cga:     <urn:holonic:ontology:> .
    @prefix schema:  <https://schema.org/> .
    @prefix dcterms: <http://purl.org/dc/terms/> .

    <urn:holon:ops> cga:dataSteward <urn:person:m-park> ;
        cga:dataOwner          <urn:person:r-jones> ;
        cga:dataClassification cga:CUI ;
        cga:repositoryOfTruthFor <urn:domain:ops> ;
        dcterms:conformsTo     <urn:ontology:schema-org> .

    <urn:person:r-jones> a schema:Person ;
        schema:name "R. Jones" ;
        schema:jobTitle "Ops Division Chief" .
''')

ds.add_boundary("urn:holon:ops", '''
    @prefix sh:     <http://www.w3.org/ns/shacl#> .
    @prefix schema: <https://schema.org/> .
    @prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .

    <urn:shapes:EventShape> a sh:NodeShape ;
        sh:targetClass schema:Event ;
        sh:property [ sh:path schema:name ;      sh:minCount 1 ; sh:datatype xsd:string ] ;
        sh:property [ sh:path schema:startDate ; sh:minCount 1 ; sh:datatype xsd:dateTime ] ;
        sh:property [ sh:path schema:location ;  sh:minCount 1 ] .
''')

print(f"Holons: {len(ds.list_holons())}")
print(f"Intel classification: cga:Secret")
print(f"Ops classification:   cga:CUI")

## Alignment Holon — semantic justification for the translation

An `AlignmentHolon` is a dedicated holon whose interior contains the
vocabulary mapping between two ontologies. It exists as a first-class
entity so the mapping itself has provenance, ownership, and versioning
independent of the portal that uses it.

This is the "governed ontology, standards, references, and ownership
underneath" the boundary contract.

In [ ]:
# ── Alignment holon: CCO sensor → Schema.org event ──

ds.add_holon("urn:holon:alignment-cco-schema", "CCO → Schema.org Alignment",
             member_of="urn:holon:org")
ds.add_interior("urn:holon:alignment-cco-schema", '''
    @prefix cga:     <urn:holonic:ontology:> .
    @prefix schema:  <https://schema.org/> .
    @prefix dcterms: <http://purl.org/dc/terms/> .
    @prefix owl:     <http://www.w3.org/2002/07/owl#> .
    @prefix skos:    <http://www.w3.org/2004/02/skos/core#> .

    <urn:alignment:cco-to-schema> a cga:AlignmentHolon ;
        rdfs:label "CCO Sensor Track → Schema.org Event" ;
        dcterms:creator    <urn:person:j-chen> ;
        dcterms:modified   "2026-04-01"^^<http://www.w3.org/2001/XMLSchema#date> ;
        dcterms:conformsTo <urn:standard:iso-11179> ;
        skos:note "Maps ISR sensor tracks to operational planning events. "
                  "Confidence threshold 0.7 applied during traversal." .

    # Mapping declarations (which CCO terms align to which Schema.org terms)
    <urn:mapping:track-to-event> a owl:Class ;
        rdfs:label "SensorTrack → Event" ;
        skos:note "cco:SensorTrack maps to schema:Event" .

    <urn:mapping:coords-to-location> a owl:DatatypeProperty ;
        rdfs:label "lat/lon → location" ;
        skos:note "CONCAT(cco:latitude, cco:longitude) maps to schema:location" .

    <urn:mapping:timestamp-to-startdate> a owl:DatatypeProperty ;
        rdfs:label "timestamp → startDate" ;
        skos:note "cco:timestamp maps to schema:startDate" .
''')

print("Alignment holon created with mapping provenance")

## The governed portal

The portal carries:
- A CONSTRUCT query that implements the translation
- `cga:usesAlignment` pointing to the alignment holon (semantic justification)
- Governance metadata on the source holon (classification, steward, conformsTo)

When this portal is traversed, the provenance chain records not just
*what* happened (PROV-O activity) but *under whose authority* and
*justified by which mapping*.

In [ ]:
# ── Governed portal: Intel → Ops ──

ds.add_portal(
    "urn:portal:intel-to-ops",
    source_iri="urn:holon:intel",
    target_iri="urn:holon:ops",
    construct_query='''
        PREFIX cco:    <urn:cco:>
        PREFIX schema: <https://schema.org/>

        CONSTRUCT {
            ?event a schema:Event ;
                schema:name ?name ;
                schema:startDate ?ts ;
                schema:location ?loc .
        }
        WHERE {
            GRAPH ?g {
                ?track a cco:SensorTrack ;
                    cco:latitude   ?lat ;
                    cco:longitude  ?lon ;
                    cco:confidence ?conf ;
                    cco:trackSource ?src ;
                    cco:timestamp  ?ts .
                FILTER(?conf >= 0.7)
                BIND(IRI(CONCAT("urn:event:", ENCODE_FOR_URI(STR(?track)))) AS ?event)
                BIND(CONCAT(?src, " track at ", STR(?lat), ",", STR(?lon)) AS ?name)
                BIND(CONCAT(STR(?lat), ",", STR(?lon)) AS ?loc)
            }
        }
    ''',
    extra_ttl='''
        @prefix cga: <urn:holonic:ontology:> .
        <urn:portal:intel-to-ops> cga:usesAlignment <urn:holon:alignment-cco-schema> .
    ''',
    label="Intel → Ops (CCO → Schema.org)",
)

print("Portal created with alignment reference")
print()

# Verify the alignment link is discoverable
rows = list(ds.query('''
    PREFIX cga: <urn:holonic:ontology:>
    SELECT ?alignment WHERE {
        GRAPH ?g {
            <urn:portal:intel-to-ops> cga:usesAlignment ?alignment .
        }
    }
'''))
print(f"Portal's alignment holon: {rows[0]['alignment']}")

## Classification boundary enforcement

The CGA shapes include `PortalClassificationShape` which warns when
a portal crosses data classification boundaries. Our portal connects
a FOUO source to a CUI target — a real classification mismatch that
the shape will flag.

In [ ]:
import pyshacl

# Load the CGA ontology + shapes for validation
ds_validated = HolonicDataset(load_ontology=True)

# Replay the same holarchy into the validated dataset
for h in ds.list_holons():
    ds_validated.add_holon(h.iri, h.label or "")
    for g_iri in h.interior_graphs:
        g = ds.backend.get_graph(g_iri)
        ds_validated.backend.put_graph(g_iri, g)
        ds_validated._register_layer(h.iri, g_iri, "hasInterior")
    for g_iri in h.boundary_graphs:
        g = ds.backend.get_graph(g_iri)
        ds_validated.backend.put_graph(g_iri, g)
        ds_validated._register_layer(h.iri, g_iri, "hasBoundary")

# Re-add portal to the validated dataset
ds_validated.add_portal(
    "urn:portal:intel-to-ops",
    source_iri="urn:holon:intel",
    target_iri="urn:holon:ops",
    construct_query="CONSTRUCT { ?s ?p ?o } WHERE { GRAPH ?g { ?s ?p ?o } }",
    extra_ttl='''
        @prefix cga: <urn:holonic:ontology:> .
        <urn:portal:intel-to-ops> cga:usesAlignment <urn:holon:alignment-cco-schema> .
    ''',
)

# Validate the registry against CGA shapes
registry = ds_validated.backend.get_graph(ds_validated.registry_iri)
shapes = ds_validated.backend.get_graph("urn:holonic:ontology:cga-shapes")

conforms, _, report_text = pyshacl.validate(registry, shacl_graph=shapes)

print(f"Registry conforms: {conforms}")
if not conforms:
    print()
    for line in report_text.split("\n"):
        if "classification" in line.lower() or "Message" in line:
            print(f"  {line.strip()}")

## Traversal with governance trail

When we traverse the governed portal, the PROV-O context graph records
the activity. The governance metadata (steward, classification,
alignment holon) is queryable alongside the traversal record because
it lives in the same holarchy — not in a separate governance system.

In [ ]:
# Traverse the governed portal
projected, membrane = ds.traverse(
    source_iri="urn:holon:intel",
    target_iri="urn:holon:ops",
    validate=True,
    agent_iri="urn:agent:ops-planner",
)

print(f"Projected {len(projected)} triples into ops interior")
print(f"Membrane health: {membrane.health.value}")
print()

# Query the projected Schema.org events
for s, p, o in projected:
    print(f"  {s}  {p.split('/')[-1]}  {o}")

In [ ]:
# Query the full governance chain in one SPARQL query:
# Who stewarded the source data? What alignment justified the
# translation? What classification levels were involved?

governance = list(ds.query('''
    PREFIX cga:     <urn:holonic:ontology:>
    PREFIX prov:    <http://www.w3.org/ns/prov#>
    PREFIX dcterms: <http://purl.org/dc/terms/>
    PREFIX schema:  <https://schema.org/>

    SELECT ?steward_name ?classification ?alignment_label ?conformsTo
    WHERE {
        GRAPH ?reg {
            <urn:holon:intel> cga:dataSteward ?steward .
            <urn:holon:intel> cga:dataClassification ?classification .

            <urn:portal:intel-to-ops> cga:usesAlignment ?alignment .
        }

        GRAPH ?g1 {
            ?steward schema:name ?steward_name .
        }

        GRAPH ?g2 {
            ?alignment rdfs:label ?alignment_label .
            ?alignment dcterms:conformsTo ?conformsTo .
        }
    }
'''))

print("Governance chain for Intel → Ops portal:")
for row in governance:
    print(f"  Steward:        {row['steward_name']}")
    print(f"  Classification: {row['classification']}")
    print(f"  Alignment:      {row['alignment_label']}")
    print(f"  Conforms to:    {row['conformsTo']}")

## What this demonstrates

The structural boundary (SHACL shapes) validates *what* can cross.
The governance metadata answers the questions underneath:

- **Which ontology governs each side?** `dcterms:conformsTo` on each
  department's interior declares `urn:ontology:cco-sensor` and
  `urn:ontology:schema-org` respectively.
- **Who owns the boundary contract?** `cga:dataSteward` and
  `cga:dataOwner` on the source holon. `cga:domainSteward` on the
  `DataDomain`.
- **What alignment justifies the translation?** `cga:usesAlignment`
  on the portal points to an `AlignmentHolon` whose interior
  contains the mapping provenance, including who created it, when,
  and under what standard.
- **What classification constraints apply?** `cga:dataClassification`
  on both holons, with `PortalClassificationShape` surfacing the
  cross-boundary mismatch.
- **What happens when the contract is violated?** SHACL validation
  against the boundary shapes; membrane health recorded in PROV-O;
  the full governance chain queryable via a single SPARQL query.

All of this metadata is first-class RDF in the holarchy. It does not
live in a separate governance system or a configuration file. It is
discoverable by the same SPARQL queries that discover holons, portals,
and traversal history.

See also:

- `04_cco_to_schemaorg` — the translation mechanics without
  governance context
- `docs/source/ontology.md` — the full CGA vocabulary including
  governance properties